# Unit stability

This notebook reviews the stored `UnitStability` selection for units that already pass the chosen `UnitCount` criteria.

The table measures amplitude drift across equal time windows and applies Hartigan's dip test with FDR correction. A unit passes only when it passes both checks.

In [ ]:
%matplotlib widget

import sys
import warnings
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure

# Silence the setuptools pkg_resources deprecation notice
warnings.filterwarnings("ignore", category=UserWarning, module="datajoint.plugin")

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "labdata_plugin").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from labdata.schema import EphysRecording, SpikeSorting, UnitCount  # noqa: E402

from labdata_plugin.schema import (  # noqa: E402
    UnitStability,
    UnitStabilityParams,
)

Choose the session and the stored quality and stability parameter sets.

In [ ]:
subject_name = "GRB058"
session_name = "20260224_152424"
liberal_criteria_id = 0
unit_criteria_id = 1
unit_stability_param_id = 0
n_bins = 50

Populate the stability table and fetch its stored unit-selection results.

In [ ]:
stability_key = {
    "subject_name": subject_name,
    "session_name": session_name,
    "unit_criteria_id": unit_criteria_id,
    "unit_stability_param_id": unit_stability_param_id,
}
UnitStability().populate(stability_key)
params = (UnitStabilityParams & stability_key).fetch1()

change_in_unit_counts = {}
probe_widget_data = {}
sorting_key_fields = [
    "subject_name",
    "session_name",
    "dataset_name",
    "probe_num",
    "parameter_set_num",
]

for master_key in (UnitStability & stability_key).fetch("KEY"):
    sorting_key = {field: master_key[field] for field in sorting_key_fields}
    analysis_df = pd.DataFrame(
        (UnitStability.Unit & master_key).fetch(
            "unit_id",
            "amplitude_drift",
            "dip_statistic",
            "dip_p_value",
            "dip_q_value",
            "dip_sample_size",
            "passes_amplitude_stability",
            "passes_unimodality",
            "passes",
            as_dict=True,
        )
    ).sort_values("unit_id")

    unit_keys = (UnitStability.Unit & master_key).fetch("KEY")
    raw_units = (SpikeSorting.Unit & unit_keys).fetch(
        "unit_id", "spike_times", "spike_amplitudes", as_dict=True
    )
    recording_duration, sampling_rate = (
        EphysRecording * EphysRecording.ProbeSetting & sorting_key
    ).fetch1("recording_duration", "sampling_rate")
    time_edges = np.linspace(
        0,
        float(recording_duration) * float(sampling_rate),
        params["n_time_windows"] + 1,
    )[1:-1]
    chunks_by_unit = {}
    for row in raw_units:
        window_index = np.digitize(row["spike_times"], time_edges)
        chunks_by_unit[row["unit_id"]] = [
            row["spike_amplitudes"][window_index == index]
            for index in range(params["n_time_windows"])
        ]
    analysis_df["chunks"] = analysis_df["unit_id"].map(chunks_by_unit)

    liberal_count = (
        UnitCount & sorting_key & {"unit_criteria_id": liberal_criteria_id}
    ).fetch1("sua")
    counts = [
        liberal_count,
        len(analysis_df),
        int(analysis_df["passes"].sum()),
    ]
    probe_num = master_key["probe_num"]
    change_in_unit_counts.setdefault(probe_num, []).append(counts)
    probe_widget_data[(session_name, probe_num)] = analysis_df

    print(
        f"{session_name} imec{probe_num}: {len(analysis_df)} quality units, "
        f"{(~analysis_df['passes_amplitude_stability'].astype(bool)).sum()} "
        "failed amplitude stability, "
        f"{(~analysis_df['passes_unimodality'].astype(bool)).sum()} "
        f"failed unimodality, {analysis_df['passes'].sum()} passed both"
    )

Compare the liberal quality count, standard quality count, and final stability-selected count.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = [
    "Liberal quality\n(criteria 0)",
    "Standard quality\n(criteria 1)",
    "Standard +\nstability selection",
]

for probe_num, counts_list in sorted(change_in_unit_counts.items()):
    counts = np.asarray(counts_list)
    for selection_counts in counts:
        ax.plot(labels, selection_counts, color="tab:blue", alpha=0.25)
    ax.plot(
        labels,
        counts.mean(axis=0),
        marker="o",
        color="tab:blue",
        label=f"imec{probe_num} mean",
    )
    total_change = 100 * (counts[:, 0].sum() - counts[:, -1].sum()) / counts[:, 0].sum()
    print(f"imec{probe_num}: total reduction = {total_change:.1f}%")

ax.set_title("Unit counts after quality and stability selection")
ax.set_ylabel("Unit count")
ax.legend()
fig.tight_layout()

Define a browser for reviewing the stored pass status and the underlying amplitude distributions.

In [ ]:
class StabilityBrowser:
    FILTER_OPTIONS = [
        "All",
        "Fails amplitude stability",
        "Fails unimodality",
        "Fails both",
        "Passes stability selection",
    ]

    def __init__(self, data, max_amplitude_drift=0.10, dip_alpha=0.05, n_bins=50):
        self.data = data
        self.max_amplitude_drift = max_amplitude_drift
        self.dip_alpha = dip_alpha
        self.n_bins = n_bins
        self._fig: Figure | None = None
        self._ax: Axes | None = None
        self._updating = False

        self.session_dropdown = widgets.Dropdown(
            options=sorted({key[0] for key in data}), description="Session:"
        )
        self.probe_dropdown = widgets.Dropdown(description="Probe:")
        self.filter_radio = widgets.RadioButtons(
            options=self.FILTER_OPTIONS, description="Show:"
        )
        self.unit_slider = widgets.SelectionSlider(
            options=["—"],
            description="Unit:",
            continuous_update=False,
            layout=widgets.Layout(width="420px"),
        )
        self.count_label = widgets.HTML()
        self.stats_html = widgets.HTML()

        self.session_dropdown.observe(self._on_session_change, names="value")
        self.probe_dropdown.observe(self._refresh_units, names="value")
        self.filter_radio.observe(self._refresh_units, names="value")
        self.unit_slider.observe(self._update_display, names="value")
        self._on_session_change()

    def _key(self):
        return (
            self.session_dropdown.value,
            int(self.probe_dropdown.value.removeprefix("imec")),
        )

    def _on_session_change(self, _change=None):
        self._updating = True
        probes = sorted(
            key[1] for key in self.data if key[0] == self.session_dropdown.value
        )
        self.probe_dropdown.options = [f"imec{probe}" for probe in probes]
        self.probe_dropdown.value = f"imec{probes[0]}"
        self._updating = False
        self._refresh_units()

    def _filtered(self):
        df = self.data[self._key()]
        selected = self.filter_radio.value
        if selected == "Fails amplitude stability":
            return df[~df["passes_amplitude_stability"].astype(bool)]
        if selected == "Fails unimodality":
            return df[~df["passes_unimodality"].astype(bool)]
        if selected == "Fails both":
            return df[
                ~df["passes_amplitude_stability"].astype(bool)
                & ~df["passes_unimodality"].astype(bool)
            ]
        if selected == "Passes stability selection":
            return df[df["passes"].astype(bool)]
        return df

    def _refresh_units(self, _change=None):
        if self._updating or not self.probe_dropdown.value:
            return
        self._updating = True
        filtered = self._filtered()
        options = filtered["unit_id"].tolist() or ["—"]
        self.unit_slider.options = options
        self.unit_slider.value = options[0]
        self.count_label.value = (
            f"<b>{len(filtered)}</b> / {len(self.data[self._key()])} units"
        )
        self._updating = False
        self._update_display()

    def _update_display(self, _change=None):
        if self._updating or self._fig is None or self._ax is None:
            return
        unit_id = self.unit_slider.value
        self._ax.clear()
        if unit_id == "—":
            self._ax.set_visible(False)
            self.stats_html.value = "<i>No units match this filter.</i>"
            self._fig.canvas.draw_idle()
            return

        self._ax.set_visible(True)
        row = self.data[self._key()].set_index("unit_id").loc[unit_id]
        drift_status = "PASS" if row.passes_amplitude_stability else "FAIL"
        dip_status = "PASS" if row.passes_unimodality else "FAIL"
        selection_status = "PASS" if row.passes else "FAIL"
        self.stats_html.value = (
            f"<b>Unit {unit_id}: {selection_status}</b><br>"
            f"Amplitude drift: {row.amplitude_drift:.3f} "
            f"(limit {self.max_amplitude_drift}) — {drift_status}<br>"
            f"Dip: {row.dip_statistic:.4f}; q = {row.dip_q_value:.4g}; "
            f"n = {row.dip_sample_size:,} (alpha {self.dip_alpha}) — {dip_status}"
        )
        for index, chunk in enumerate(row.chunks):
            self._ax.hist(
                chunk,
                bins=self.n_bins,
                histtype="step",
                alpha=0.8,
                label=f"Time {index + 1}",
            )
        self._ax.set(title=f"Unit {unit_id}", xlabel="Spike amplitude", ylabel="Count")
        self._ax.legend(fontsize=8)
        self._fig.tight_layout()
        self._fig.canvas.draw_idle()

    def show(self):
        with plt.ioff():
            self._fig, self._ax = plt.subplots(figsize=(5, 3.5))
        controls = widgets.VBox(
            [self.session_dropdown, self.probe_dropdown, self.filter_radio]
        )
        details = widgets.VBox(
            [self.unit_slider, self.count_label, self.stats_html],
            layout=widgets.Layout(margin="0 0 0 20px"),
        )
        display(widgets.HBox([widgets.VBox([controls, self._fig.canvas]), details]))
        self._refresh_units()

Open the interactive stability-selection browser.

In [ ]:
browser = StabilityBrowser(
    probe_widget_data,
    max_amplitude_drift=params["max_amplitude_drift"],
    dip_alpha=params["dip_alpha"],
    n_bins=n_bins,
)
browser.show()